<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/multiomics/notebooks/02_multiomics_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕸️ Multi-omics II — Networks, pathways and a mechanism

---

In Multi-omics I we integrated the proteome and the metabolome at the level of *patients*
— who resembles whom. This session integrates at the level of *molecules*: which protein
goes with which metabolite, and what that says about metabolism.

This is where the three days come together. We use the networks you learned this morning,
the enrichment logic from Days 2 and 3, and both matrices at once — and we end with a
specific, testable biological statement about carbapenem-resistant sepsis.

### What you will be able to do afterwards

1. Build and interpret a **cross-omics correlation network**.
2. Run a **joint pathway analysis** over proteins and metabolites in one KEGG vocabulary.
3. Export a multi-omics network to **Cytoscape** with the attributes that make it readable.
4. Judge honestly whether adding a second omics layer improves a **classifier**.

## 🧬 Why a network?

A correlation between one protein and one metabolite is a fact about two molecules. A
*network* of such correlations is a statement about organisation: which molecules move
together, which of them sit at the centre of everything, and whether the modules that
emerge look like biochemistry we recognise.

For proteins and metabolites the interpretation is unusually concrete, because the
relationship is often mechanical:

- an **enzyme** and its **substrate** or **product**;
- a **transporter** and the molecule it moves;
- a **carrier** (albumin, lipoproteins) and its cargo;
- or both simply responding to the same upstream driver — inflammation, organ failure,
  a drug.

The last of these is the most common and the least interesting, which is why a network is
a way of *generating* hypotheses, never of confirming them.

⚠️ Two warnings, stated plainly:

**Correlation across patients is not a reaction.** An enzyme–substrate pair need not
correlate positively; if the enzyme consumes the substrate, more enzyme could mean *less*
substrate. Sign is not direction.

**The number of tests is enormous.** A few hundred proteins against a few hundred
metabolites is tens of thousands of correlations. Without multiple-testing control you
will find a beautiful, entirely spurious network. We control it, and we look at how much
difference that makes.

In [ ]:
%pip install -q networkx acore

In [ ]:
import itertools
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

OUT_DIR = Path("multiomics/results")
EXPORT_DIR = OUT_DIR / "cytoscape"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
GROUP_COLOURS = {"Con": "#4C72B0", "CSKP": "#DD8452", "CRKP": "#C44E52"}
GROUP_ORDER = ["Con", "CSKP", "CRKP"]

## 1. The two views, again

Same preparation as this afternoon, repeated here so this notebook stands on its own.

In [ ]:
proteins_raw = pd.read_csv(f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv", sep="\t")
metabolites_raw = pd.read_csv(f"{BASE_URL}/metabolomics/data/metabolite_matrix.tsv", sep="\t")
metadata = pd.read_csv(f"{BASE_URL}/metadata/sample_metadata.tsv", sep="\t").set_index("sample_id")
metabolite_annotation = pd.read_csv(
    f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv", sep="\t"
).set_index("metabolite")

protein_info = proteins_raw[["protein_group", "genes", "description"]].set_index("protein_group")
shared = [s for s in metadata.index
          if s in proteins_raw.columns and s in metabolites_raw.columns]
groups = metadata.loc[shared, "group"]
print(f"{len(shared)} patients measured on both platforms")


def prepare_view(wide, feature_col, samples, max_missing=0.30):
    matrix = np.log2(wide.set_index(feature_col)[samples].T)
    matrix = matrix.loc[:, matrix.isna().mean() <= max_missing]
    matrix = matrix.fillna(matrix.min())
    matrix = (matrix - matrix.mean()) / matrix.std()
    return matrix.loc[:, matrix.std() > 0]


proteomics = prepare_view(proteins_raw, "protein_group", shared)
metabolomics = prepare_view(metabolites_raw, "metabolite", shared)


def gene_label(protein_group: str) -> str:
    """Prefer a gene symbol for display, fall back to the accession."""
    if protein_group in protein_info.index:
        genes = protein_info.loc[protein_group, "genes"]
        if isinstance(genes, str) and genes:
            return genes.split(";")[0]
    return protein_group


print(f"proteomics : {proteomics.shape}")
print(f"metabolomics: {metabolomics.shape}")

### Trimming to a network we can look at

All-against-all would be hundreds of thousands of tests and a hairball nobody can read.
We keep the most **variable** features in each layer — variance is a reasonable, unbiased
way to say "this molecule does something across these patients", and it does not use the
group labels, so it cannot inflate the group differences we look at later.

In [ ]:
N_PROTEINS, N_METABOLITES = 200, 200

top_proteins = proteomics.var().nlargest(N_PROTEINS).index
top_metabolites = metabolomics.var().nlargest(N_METABOLITES).index
X_prot = proteomics[top_proteins]
X_metab = metabolomics[top_metabolites]

print(f"{len(top_proteins)} proteins x {len(top_metabolites)} metabolites "
      f"= {len(top_proteins) * len(top_metabolites):,} correlations to test")

## 2. The cross-omics correlation network

We use **Spearman** correlation: it responds to any monotone relationship, not only a
linear one, and it is far less easily hijacked by a single extreme patient — which
matters when n = 45 and some of these patients are in multi-organ failure.

Then **Benjamini–Hochberg** across all pairs, and we keep edges that are both significant
and reasonably strong.

In [ ]:
rho, pval = stats.spearmanr(X_prot.values, X_metab.values)
# spearmanr on two matrices returns the full (p+m) x (p+m) matrix; we want the cross block
n_p = X_prot.shape[1]
rho_cross = pd.DataFrame(rho[:n_p, n_p:], index=X_prot.columns, columns=X_metab.columns)
p_cross = pd.DataFrame(pval[:n_p, n_p:], index=X_prot.columns, columns=X_metab.columns)

edges = (
    rho_cross.stack().rename("rho").to_frame()
    .join(p_cross.stack().rename("pvalue"))
    .rename_axis(["protein_group", "metabolite"])
    .reset_index()
)
# Benjamini-Hochberg, written out
edges = edges.sort_values("pvalue").reset_index(drop=True)
edges["padj"] = (edges["pvalue"] * len(edges) / (edges.index + 1)).clip(upper=1)
edges["padj"] = edges["padj"][::-1].cummin()[::-1]

print(f"{len(edges):,} protein-metabolite pairs tested")
print(f"nominally significant (p < 0.05)     : {(edges['pvalue'] < 0.05).sum():,}")
print(f"after FDR correction (padj < 0.05)   : {(edges['padj'] < 0.05).sum():,}")
edges.head()

⚙️ Compare the two counts with what chance alone would give. Chance would put about 5 % of
the tests below p = 0.05. If the nominal count is clearly above that, there is real
cross-layer structure; the corrected count tells you how much of it you can point at
individually.

### Choosing the threshold, out loud

With 45 patients the strongest correlation we can possibly achieve is limited: even a
perfect monotone relationship in 45 points only reaches so small a p-value, and after
dividing by tens of thousands of tests, **a 5 % FDR may leave almost nothing**. That is a
genuine consequence of sample size, not a bug, and pretending otherwise is how
multi-omics networks get published with invisible error rates.

So we make the trade-off explicitly. We use **FDR < 0.20** together with a
**|rho| ≥ 0.45** floor, and we state what that buys and costs: roughly one edge in five is
expected to be false, which is acceptable for a network we will use to *generate*
hypotheses and unacceptable for a network we would report as a set of findings. Exercise 1
asks you to tighten it and watch what happens.

In [ ]:
RHO_CUTOFF, PADJ_CUTOFF = 0.45, 0.20
selected = edges[(edges["padj"] < PADJ_CUTOFF) & (edges["rho"].abs() >= RHO_CUTOFF)].copy()
selected["sign"] = np.where(selected["rho"] > 0, "positive", "negative")
print(f"edges kept: {len(selected)} "
      f"({(selected['sign'] == 'positive').sum()} positive, "
      f"{(selected['sign'] == 'negative').sum()} negative)")
print(f"expected false edges at FDR {PADJ_CUTOFF:.0%}: about {PADJ_CUTOFF * len(selected):.0f}")
for cutoff in (0.05, 0.10, 0.20):
    n = ((edges["padj"] < cutoff) & (edges["rho"].abs() >= RHO_CUTOFF)).sum()
    print(f"  at FDR < {cutoff:.2f}: {n} edges")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
axes[0].hist(edges["rho"], bins=80, color="#9AA5B1")
for cut in (-RHO_CUTOFF, RHO_CUTOFF):
    axes[0].axvline(cut, color="#C44E52", ls="--")
axes[0].set(xlabel="Spearman rho (protein vs metabolite)", ylabel="pairs",
            title="All cross-omics correlations")
axes[1].scatter(edges["rho"], -np.log10(edges["pvalue"]), s=3, alpha=0.2, color="#9AA5B1")
axes[1].scatter(selected["rho"], -np.log10(selected["pvalue"]), s=8, alpha=0.7, color="#C44E52")
axes[1].set(xlabel="Spearman rho", ylabel="-log10 p-value", title="Selected edges in red")
fig.tight_layout()

### Building the graph

The result is a **bipartite** network: two kinds of node, and edges only ever between the
two kinds. Bipartite structure is worth preserving explicitly — it lets us ask, for
example, which metabolite has the most protein partners.

In [ ]:
graph = nx.Graph()
for protein in selected["protein_group"].unique():
    graph.add_node(f"P:{protein}", layer="protein", label=gene_label(protein),
                   accession=protein,
                   description=str(protein_info.loc[protein, "description"])[:80]
                   if protein in protein_info.index else "")
for metabolite in selected["metabolite"].unique():
    row = metabolite_annotation.loc[metabolite] if metabolite in metabolite_annotation.index else None
    graph.add_node(f"M:{metabolite}", layer="metabolite", label=metabolite,
                   compound_class=str(row["class_i"]) if row is not None else "",
                   kegg_compound=str(row["kegg_compound"]) if row is not None else "")
for _, edge in selected.iterrows():
    graph.add_edge(f"P:{edge['protein_group']}", f"M:{edge['metabolite']}",
                   rho=float(edge["rho"]), abs_rho=abs(float(edge["rho"])),
                   padj=float(edge["padj"]), sign=edge["sign"])

print(nx.info(graph) if hasattr(nx, "info") else
      f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"protein nodes    : {sum(1 for _, d in graph.nodes(data=True) if d['layer'] == 'protein')}")
print(f"metabolite nodes : {sum(1 for _, d in graph.nodes(data=True) if d['layer'] == 'metabolite')}")
print(f"connected components: {nx.number_connected_components(graph)}")

### Who are the hubs?

In a bipartite cross-omics network a hub means something specific: a protein correlated
with *many* metabolites is either a genuine metabolic controller, or — much more often —
a protein that tracks a global state such as inflammation or liver function, so that it
correlates with everything that also tracks that state. Read hubs with suspicion.

In [ ]:
degree = pd.Series(dict(graph.degree())).sort_values(ascending=False)
node_layer = pd.Series(nx.get_node_attributes(graph, "layer"))
node_label = pd.Series(nx.get_node_attributes(graph, "label"))

hub_table = pd.DataFrame({"degree": degree, "layer": node_layer, "label": node_label}).dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, layer in zip(axes, ["protein", "metabolite"]):
    top = hub_table[hub_table["layer"] == layer].head(12).iloc[::-1]
    ax.barh(top["label"].astype(str).str.slice(0, 30), top["degree"],
            color="#4C72B0" if layer == "protein" else "#DD8452")
    ax.set(xlabel="number of partners in the other layer", title=f"{layer} hubs")
fig.tight_layout()
fig.savefig(OUT_DIR / "cross_omics_hubs.png", bbox_inches="tight", dpi=200)
hub_table.head(15)

### Modules

Community detection asks whether the network breaks into groups of nodes that are more
connected to each other than to the rest. In a bipartite network each community should
contain both proteins and metabolites — and a community that mixes the layers is exactly
the cross-layer module we came for.

In [ ]:
communities = nx.community.louvain_communities(graph, weight="abs_rho", seed=42)
membership = {node: i for i, community in enumerate(communities) for node in community}
nx.set_node_attributes(graph, membership, "community")

summary = []
for i, community in enumerate(communities):
    proteins_in = [node_label[n] for n in community if node_layer[n] == "protein"]
    metabolites_in = [node_label[n] for n in community if node_layer[n] == "metabolite"]
    if len(community) < 4:
        continue
    summary.append({
        "community": i, "size": len(community),
        "proteins": len(proteins_in), "metabolites": len(metabolites_in),
        "example proteins": ", ".join(sorted(proteins_in)[:5]),
        "example metabolites": ", ".join(sorted(metabolites_in)[:4]),
    })
community_summary = pd.DataFrame(summary)
if len(community_summary):
    community_summary = community_summary.sort_values("size", ascending=False)
print(f"{len(communities)} communities, {len(community_summary)} with at least 4 members")
community_summary

### Drawing it

A bipartite layout — proteins in one column, metabolites in the other — is far more
readable than a force-directed hairball when you want to see *which* protein connects to
*which* metabolite.

In [ ]:
largest = max(communities, key=len)
subgraph = graph.subgraph(largest)
protein_nodes = [n for n in subgraph if subgraph.nodes[n]["layer"] == "protein"]

positions = nx.bipartite_layout(subgraph, protein_nodes, align="vertical", scale=2)
edge_colours = ["#C44E52" if subgraph.edges[e]["rho"] > 0 else "#4C72B0" for e in subgraph.edges]
edge_widths = [2.5 * subgraph.edges[e]["abs_rho"] for e in subgraph.edges]

fig, ax = plt.subplots(figsize=(11, max(5, 0.32 * len(subgraph))))
nx.draw_networkx_edges(subgraph, positions, ax=ax, edge_color=edge_colours,
                       width=edge_widths, alpha=0.6)
nx.draw_networkx_nodes(subgraph, positions, ax=ax, nodelist=protein_nodes,
                       node_color="#4C72B0", node_shape="s", node_size=260,
                       edgecolors="white", label="protein")
metabolite_nodes = [n for n in subgraph if n not in protein_nodes]
nx.draw_networkx_nodes(subgraph, positions, ax=ax, nodelist=metabolite_nodes,
                       node_color="#DD8452", node_shape="o", node_size=260,
                       edgecolors="white", label="metabolite")
nx.draw_networkx_labels(subgraph, positions, ax=ax, font_size=7,
                        labels={n: str(subgraph.nodes[n]["label"])[:24] for n in subgraph})
ax.set_title("Largest cross-omics module — red edges positive, blue negative")
ax.legend(scatterpoints=1, loc="lower right")
ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "cross_omics_module.png", bbox_inches="tight", dpi=200)

## 3. Exporting to Cytoscape

For anything beyond a quick look, [Cytoscape](https://cytoscape.org/) is the right tool:
interactive layout, styling by attribute, and dozens of apps (StringApp, MetScape,
ClueGO) that pull in external knowledge.

We export two things, because Cytoscape is happiest with both:

1. a **GraphML** file — topology *and* attributes in one file;
2. an **edge table** plus a **node attribute table** as CSV — the route that always works,
   and the one you will use when your attributes come from elsewhere.

In [ ]:
nx.set_node_attributes(graph, {n: float(d) for n, d in graph.degree()}, "degree")
nx.set_node_attributes(graph, nx.betweenness_centrality(graph), "betweenness")

graphml_path = EXPORT_DIR / "cross_omics_network.graphml"
nx.write_graphml(graph, graphml_path)

node_table = pd.DataFrame.from_dict(dict(graph.nodes(data=True)), orient="index")
node_table.index.name = "node"
node_table.to_csv(EXPORT_DIR / "node_attributes.csv")

edge_table = selected.assign(
    source=lambda d: "P:" + d["protein_group"],
    target=lambda d: "M:" + d["metabolite"],
)[["source", "target", "rho", "abs_rho" if "abs_rho" in selected else "rho", "padj", "sign"]]
edge_table.columns = ["source", "target", "rho", "abs_rho", "padj", "sign"]
edge_table["abs_rho"] = edge_table["rho"].abs()
edge_table.to_csv(EXPORT_DIR / "edge_table.csv", index=False)

!ls -lh multiomics/results/cytoscape
node_table.head()

### Loading it in Cytoscape

**Option A — the GraphML file**
`File → Import → Network from File…` and choose `cross_omics_network.graphml`. Topology and
all node/edge attributes arrive together.

**Option B — the tables**
1. `File → Import → Network from File…` → `edge_table.csv`; set `source` as *Source Node*,
   `target` as *Target Node*, and the rest as *Edge Attribute*.
2. `File → Import → Table from File…` → `node_attributes.csv`; key column `node`, matched
   against `shared name`.

**Then style it** (`Style` panel) — the mapping is what turns a hairball into a figure:

| Visual property | Map to | Mapping type |
|---|---|---|
| Node Label | `label` | Passthrough |
| Node Shape | `layer` | Discrete — square for protein, ellipse for metabolite |
| Node Fill Colour | `community` | Discrete |
| Node Size | `degree` | Continuous, 20 → 60 |
| Edge Stroke Colour | `rho` | Continuous, diverging blue–white–red centred on 0 |
| Edge Width | `abs_rho` | Continuous, 1 → 6 |

Layout: `Layout → Prefuse Force Directed` for an overview, or
`Layout → yFiles Hierarchic` to show the two layers as two tiers. Export with
`File → Export → Network to Image…` as PDF or SVG so the figure stays sharp.

> 💻 If Cytoscape is running on the same machine, `py4cytoscape` lets you drive all of the
> above from Python (`p4c.create_network_from_networkx(graph)`). Convenient, but it needs a
> live Cytoscape; the files above always work.

## 4. Joint pathway analysis

The network told us which molecules travel together. Pathway analysis asks whether they
belong to the same *known* biochemistry — and here multi-omics pays a concrete dividend,
because **KEGG annotates both genes and compounds against the same pathway maps**. One
vocabulary, two layers.

The recipe:

1. proteins → UniProt accession → KEGG gene → KEGG pathway
2. metabolites → KEGG compound → KEGG pathway
3. per pathway, count hits from each layer and test against the measured background

In [ ]:
def fetch_kegg_mappings() -> dict | None:
    """Fetch gene/compound-to-pathway mappings from the KEGG REST API."""
    try:
        uniprot_to_kegg = pd.read_csv("https://rest.kegg.jp/conv/hsa/uniprot", sep="\t",
                                      header=None, names=["uniprot", "kegg_gene"])
        gene_pathways = pd.read_csv("https://rest.kegg.jp/link/pathway/hsa", sep="\t",
                                    header=None, names=["kegg_gene", "pathway"])
        compound_pathways = pd.read_csv("https://rest.kegg.jp/link/pathway/cpd", sep="\t",
                                        header=None, names=["compound", "pathway"])
        pathway_names = pd.read_csv("https://rest.kegg.jp/list/pathway/hsa", sep="\t",
                                    header=None, names=["pathway", "name"])
        return {
            "uniprot_to_kegg": uniprot_to_kegg,
            "gene_pathways": gene_pathways,
            "compound_pathways": compound_pathways,
            "pathway_names": pathway_names,
        }
    except Exception as error:
        print("KEGG API not reachable:", error)
        return None


kegg = fetch_kegg_mappings()
KEGG_LIVE = kegg is not None
if KEGG_LIVE:
    for name, table in kegg.items():
        print(f"{name:20s} {len(table):>8,} rows")

> ⚙️ `acore.io.kegg` wraps the same endpoints (`link_kegg_batch`,
> `parse_compound_pathway_mapping`, `fetch_kegg_ko_descriptions`) with batching and
> caching, which you want for anything larger than this.

KEGG pathway identifiers come in two flavours: `hsa00270` is the *human* instance of a
pathway, `map00270` the reference version. Compound links use `map`, gene links use `hsa`,
and the numeric part is shared — so we normalise to the number and can then join the two
layers.

In [ ]:
def normalise_pathway(value: str) -> str:
    """'path:hsa00270' / 'path:map00270' / 'ko00270' -> '00270'."""
    text = str(value).replace("path:", "")
    digits = "".join(ch for ch in text if ch.isdigit())
    return digits


if KEGG_LIVE:
    up_map = (
        kegg["uniprot_to_kegg"]
        .assign(uniprot=lambda d: d["uniprot"].str.replace("up:", "", regex=False))
        .drop_duplicates(subset=["uniprot"], keep="first") # Added to handle duplicate uniprot IDs
        .set_index("uniprot")["kegg_gene"]
    )
    gene_to_pathway = kegg["gene_pathways"].assign(
        code=lambda d: d["pathway"].map(normalise_pathway))
    protein_pathways = (
        pd.DataFrame({"protein_group": list(proteomics.columns)})
        .assign(accession=lambda d: d["protein_group"].str.split(";").str[0])
        .assign(kegg_gene=lambda d: d["accession"].map(up_map))
        .dropna(subset=["kegg_gene"])
        .merge(gene_to_pathway[["kegg_gene", "code"]], on="kegg_gene")
        [["protein_group", "code"]].drop_duplicates()
    )
    compound_to_pathway = kegg["compound_pathways"].assign(
        compound=lambda d: d["compound"].str.replace("cpd:", "", regex=False),
        code=lambda d: d["pathway"].map(normalise_pathway),
    )
    metabolite_pathways = (
        metabolite_annotation["kegg_compound"].dropna().rename("compound").reset_index()
        .merge(compound_to_pathway[["compound", "code"]], on="compound")
        [["metabolite", "code"]].drop_duplicates()
    )
    pathway_names = (
        kegg["pathway_names"]
        .assign(code=lambda d: d["pathway"].map(normalise_pathway))
        .drop_duplicates("code").set_index("code")["name"]
        .str.replace(r" - Homo sapiens.*", "", regex=True)
    )
    print(f"{protein_pathways['protein_group'].nunique()} proteins mapped to "
          f"{protein_pathways['code'].nunique()} pathways")
    print(f"{metabolite_pathways['metabolite'].nunique()} metabolites mapped to "
          f"{metabolite_pathways['code'].nunique()} pathways")
else:
    protein_pathways = metabolite_pathways = pathway_names = None
    print("Skipping the live joint analysis; the published result is shown below instead.")

### Testing pathways with both layers at once

For each pathway we ask: among the molecules we *measured* that belong to it, are more of
them in our differential lists than chance would predict? We run the test on the union of
the two layers, so a pathway supported weakly by proteins *and* weakly by metabolites can
reach significance — which is the whole point.

In [ ]:
if KEGG_LIVE:
    # differential features, from the two single-omics sessions' criteria
    protein_hits = set()
    metabolite_hits = set()

    crkp, cskp = groups[groups == "CRKP"].index, groups[groups == "CSKP"].index
    for name, matrix, target in [("protein", proteomics, protein_hits),
                                 ("metabolite", metabolomics, metabolite_hits)]:
        for feature in matrix.columns:
            a, b = matrix.loc[crkp, feature], matrix.loc[cskp, feature]
            if stats.ttest_ind(a, b, equal_var=True).pvalue < 0.05 and abs(a.mean() - b.mean()) > 0.4:
                target.add(feature)
    print(f"differential proteins: {len(protein_hits)}, metabolites: {len(metabolite_hits)}")

    annotation_long = pd.concat([
        protein_pathways.rename(columns={"protein_group": "feature"}).assign(layer="protein"),
        metabolite_pathways.rename(columns={"metabolite": "feature"}).assign(layer="metabolite"),
    ])
    background = set(proteomics.columns) | set(metabolomics.columns)
    foreground = protein_hits | metabolite_hits
    annotation_long = annotation_long[annotation_long["feature"].isin(background)]

    rows = []
    for code, block in annotation_long.groupby("code"):
        members = set(block["feature"])
        hits = members & foreground
        if len(members) < 3 or not hits:
            continue
        table = [[len(hits), len(foreground) - len(hits)],
                 [len(members) - len(hits), len(background - foreground) - (len(members) - len(hits))]]
        rows.append({
            "code": code,
            "pathway": pathway_names.get(code, code),
            "measured": len(members),
            "hits": len(hits),
            "protein_hits": len(hits & protein_hits),
            "metabolite_hits": len(hits & metabolite_hits),
            "pvalue": stats.fisher_exact(table, alternative="greater")[1],
            "features": ", ".join(sorted(gene_label(h) if h in protein_info.index else h
                                         for h in hits)[:6]),
        })
    joint = pd.DataFrame(rows).sort_values("pvalue")
    joint["padj"] = (joint["pvalue"] * len(joint) / np.arange(1, len(joint) + 1)).clip(upper=1)
    both_layers = joint[(joint["protein_hits"] > 0) & (joint["metabolite_hits"] > 0)]
    print(f"{len(joint)} pathways tested, {len(both_layers)} supported by BOTH layers")
    display(joint.head(20))

In [ ]:
if KEGG_LIVE and len(both_layers):
    top = both_layers.head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9.5, 4.6))
    ax.barh(top["pathway"].astype(str).str.slice(0, 45), top["protein_hits"],
            color="#4C72B0", label="proteins")
    ax.barh(top["pathway"].astype(str).str.slice(0, 45), top["metabolite_hits"],
            left=top["protein_hits"], color="#DD8452", label="metabolites")
    ax.set(xlabel="differential features in the pathway",
           title="Pathways supported by both omics layers")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_DIR / "joint_pathway_analysis.png", bbox_inches="tight", dpi=200)

### The published integrated result

The authors ran the same kind of analysis in MetaboAnalyst. Here is what they found —
three pathways, and note how few features each rests on.

In [ ]:
published_integrated = pd.read_csv(
    f"{BASE_URL}/multiomics/data/published_integrated_pathways.tsv", sep="\t"
)
published_integrated

🧬 Read the `matched_features` column carefully: `hsa:27430; cpd:C00019; cpd:C00022`. That
is **one gene and two compounds** — `hsa:27430` is *MAT2B*, `C00019` is
S-adenosyl-L-methionine (SAM) and `C00022` is pyruvate. A three-feature pathway hit with
an FDR of 0.57 is, on its own, weak evidence, and the paper says so.

But it is a *specific* hypothesis rather than a vague one, and specific hypotheses can be
checked. Which is what we do next — with the actual measurements rather than the
enrichment statistic.

## 5. 🧬 The methionine cycle, in one cohort

This is the payoff of three days. Here is the biochemistry the paper points at:

```
           MAT2A / MAT2B                    methyltransferases
 methionine ──────────────► SAM ─────────────────────────────► SAH
     ▲                                                          │  AHCY
     │  remethylation                                           ▼
     │  (betaine, folate, choline)                        homocysteine
     │                                                          │  transsulfuration
     └──────────────────────────────────────────────────────────┤
                                                                ▼
                                                   cysteine ──► glutathione
```

Every arrow needs an enzyme (a **protein**) and moves a **metabolite** — a textbook case
where one omics layer cannot tell the story. Three things make it interesting in sepsis:

- **SAM** is the cell's universal methyl donor: the SAM/SAH ratio is a direct index of
  methylation capacity, and it drops under metabolic stress.
- **Glutathione**, made from cysteine downstream, is the main intracellular antioxidant.
  Septic patients burn through it.
- **Methionine** is contested territory: the host needs it, and so does the bacterium.

And we measured them. Let us look.

In [ ]:
CYCLE_PROTEINS = {"MAT2A": "methionine adenosyltransferase, catalytic",
                  "MAT2B": "methionine adenosyltransferase, regulatory",
                  "AHCY": "S-adenosylhomocysteine hydrolase"}
CYCLE_METABOLITES = ["L-Methionine", "S-Adenosyl-L-Methionine", "S-(5-Adenosy)-L-Homocysteine",
                     "DL-Homocysteine", "L-Cysteine", "Glutathione Oxidized",
                     "Betaine", "Choline", "Sarcosine", "Taurine", "L-Serine"]

gene_to_accession = {}
for accession, row in protein_info.iterrows():
    genes = row["genes"]
    if isinstance(genes, str):
        for gene in genes.split(";"):
            gene_to_accession.setdefault(gene, accession)

# use the full (unfiltered) matrices here: these are exactly the low-abundance,
# mechanistically interesting proteins that a completeness filter throws away
full_proteins = np.log2(proteins_raw.set_index("protein_group")[shared].T)
full_metabolites = np.log2(metabolites_raw.set_index("metabolite")[shared].T)

available_proteins = {g: gene_to_accession[g] for g in CYCLE_PROTEINS
                      if g in gene_to_accession and gene_to_accession[g] in full_proteins.columns}
available_metabolites = [m for m in CYCLE_METABOLITES if m in full_metabolites.columns]

print("proteins of the cycle measured   :", list(available_proteins))
print("metabolites of the cycle measured:", available_metabolites)

In [ ]:
cycle = pd.DataFrame(index=shared)
for gene, accession in available_proteins.items():
    cycle[gene] = full_proteins[accession]
for metabolite in available_metabolites:
    cycle[metabolite] = full_metabolites[metabolite]
cycle["group"] = groups

# the methylation index: SAM / SAH, on the log scale a difference
if {"S-Adenosyl-L-Methionine", "S-(5-Adenosy)-L-Homocysteine"} <= set(cycle.columns):
    cycle["SAM/SAH (log2 ratio)"] = (
        cycle["S-Adenosyl-L-Methionine"] - cycle["S-(5-Adenosy)-L-Homocysteine"]
    )

panels = [c for c in cycle.columns if c != "group"]
long = cycle.melt(id_vars="group", value_vars=panels, var_name="feature", value_name="log2 level")
grid = sns.catplot(
    data=long, x="group", y="log2 level", hue="group", col="feature", col_wrap=4,
    order=GROUP_ORDER, hue_order=GROUP_ORDER, palette=GROUP_COLOURS, kind="box",
    height=2.5, aspect=0.9, showfliers=False, legend=False, sharey=False,
)
for ax in grid.axes.flat:
    ax.set_title(ax.get_title().split(" = ")[-1][:26], fontsize=9)
grid.figure.suptitle("The methionine cycle across Con -> CSKP -> CRKP", y=1.02)
grid.figure.savefig(OUT_DIR / "methionine_cycle.png", bbox_inches="tight", dpi=200)

In [ ]:
# Formal tests: CRKP vs CSKP, and a monotone trend across the severity axis
order_values = metadata.loc[shared, "group_order"]
rows = []
for feature in panels:
    values = cycle[feature]
    crkp_values = values[groups == "CRKP"].dropna()
    cskp_values = values[groups == "CSKP"].dropna()
    trend = stats.spearmanr(values, order_values, nan_policy="omit")
    rows.append({
        "feature": feature,
        "n observed": values.notna().sum(),
        "log2FC CRKP-CSKP": crkp_values.mean() - cskp_values.mean(),
        "p (CRKP vs CSKP)": stats.ttest_ind(crkp_values, cskp_values, equal_var=True).pvalue,
        "rho (trend)": trend.statistic,
        "p (trend)": trend.pvalue,
    })
cycle_stats = pd.DataFrame(rows).sort_values("p (trend)")
cycle_stats.round(4)

### Does MAT2B behave like an enzyme of this cycle?

The sharpest test of the paper's hypothesis: if MAT2B abundance is functionally connected
to methionine metabolism in these patients, it should correlate with the cycle's
metabolites — and above all with SAM, its product.

In [ ]:
if "MAT2B" in cycle.columns:
    partners = [c for c in panels if c != "MAT2B"]
    rows = []
    for partner in partners:
        valid = cycle[["MAT2B", partner]].dropna()
        if len(valid) < 15:
            continue
        result = stats.spearmanr(valid["MAT2B"], valid[partner])
        rows.append({"partner": partner, "n": len(valid),
                     "rho": result.statistic, "pvalue": result.pvalue})
    mat2b = pd.DataFrame(rows).sort_values("pvalue")
    display(mat2b.round(4))

    best = mat2b.iloc[0]["partner"]
    valid = cycle[["MAT2B", best, "group"]].dropna()
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    for group in GROUP_ORDER:
        sub = valid[valid["group"] == group]
        ax.scatter(sub["MAT2B"], sub[best], label=group, s=65,
                   color=GROUP_COLOURS[group], edgecolor="white")
    slope, intercept, r, p, _ = stats.linregress(valid["MAT2B"], valid[best])
    xs = np.linspace(valid["MAT2B"].min(), valid["MAT2B"].max(), 50)
    ax.plot(xs, intercept + slope * xs, color="grey", ls="--")
    ax.set(xlabel="MAT2B (log2 intensity)", ylabel=f"{best} (log2 level)",
           title=f"MAT2B vs {best}\nSpearman rho = {mat2b.iloc[0]['rho']:.2f}, "
                 f"p = {mat2b.iloc[0]['pvalue']:.3g}, n = {int(mat2b.iloc[0]['n'])}")
    ax.legend(title="group", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "mat2b_correlation.png", bbox_inches="tight", dpi=200)

⚖️ **Now be a scientist about it.** Whatever the numbers came out as, hold them against
these three objections:

1. **n is small and MAT2B is sparsely measured.** A correlation estimated on a couple of
   dozen patients has a wide confidence interval. Compute it before you believe the point
   estimate.
2. **Serum is not tissue.** MAT2B is an intracellular protein; if it is in serum, it
   probably leaked from damaged cells. Its abundance may report **cell death**, not
   methionine metabolism — and cell death correlates with everything in sepsis.
3. **The pathway hit was FDR 0.57.** We came to this pathway *because* the paper pointed
   at it. Testing it here is a coherence check, not independent confirmation.

What would settle it: measuring the cycle in tissue or leukocytes, an intervention that
changes SAM availability, or simply another cohort. That is what "further validation is
needed" means when it is not a throwaway phrase.

## 6. Where we have arrived

| Day | What we did | The lesson |
|---|---|---|
| 1 | Raw spectra → matrices, with Nextflow pipelines | preprocessing is a chain of decisions, not a black box |
| 2 | Python, pandas, plotting; the serum proteome tested | a differential list is data *plus* analyst choices |
| 3 | The metabolome; integration; networks and pathways | integration pays off where mechanism crosses layers |

And the biology: septic patients infected with a **carbapenem-resistant** *K. pneumoniae*
show a serum signature involving coagulation, antigen presentation and — where the two
layers meet — **one-carbon and methionine metabolism**. It is a hypothesis with a specific
mechanism and a specific way to be wrong. That is what a good multi-omics analysis
produces: not certainty, but something worth the next experiment.

## 📚 Further reading

- Yugi K *et al.* (2016) *Trans-omics: how to reconstruct biochemical networks across
  multiple omic layers.* Trends Biotechnol 34:276–290.
- Chong J & Xia J (2018) *MetaboAnalystR: an R package for flexible and reproducible
  analysis of metabolomics data.* Bioinformatics 34:4313–4314. — the joint pathway analysis
  the paper used.
- Shannon P *et al.* (2003) *Cytoscape: a software environment for integrated models of
  biomolecular interaction networks.* Genome Res 13:2498–2504.
- Barabási AL, Gulbahce N, Loscalzo J (2011) *Network medicine: a network-based approach
  to human disease.* Nat Rev Genet 12:56–68.
- Lu SC & Mato JM (2012) *S-adenosylmethionine in liver health, injury, and cancer.*
  Physiol Rev 92:1515–1542. — the biology behind section 5.